In [0]:
%pip install prophet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 131.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 69.9 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import mlflow
import mlflow.prophet
from mlflow.models import infer_signature
from prophet import Prophet
import pandas as pd

# 1. Load fully engineered Gold data!
df_gold = spark.table("workspace.default.gold_power_demand").toPandas()

# 2. Prophet strictly requires the timestamp to be named 'ds' and the target to be 'y'
# Assuming  target is 'demand_mw' based on  raw data
df_train = df_gold.rename(columns={'timestamp': 'ds', 'demand_mw': 'y'})

# 3. Train the Model natively in Python 3.12
# We turn off default daily/weekly seasonality because we can pass  engineered features instead
model = Prophet()

# Add the regressors (features) that you already built into  Gold table
model.add_regressor('temperature') # From  weather data
model.add_regressor('hour')        # From  time intelligence
model.add_regressor('day_of_week') # From  time intelligence

# Fit the model
model.fit(df_train)

# 1. Grab a small sample of your training data
sample_input = df_train.head(5)

# 2. Generate a sample prediction to show UC what the output looks like
sample_output = model.predict(sample_input)

# 3. Auto-infer the signature (the "contract")
signature = infer_signature(model_input=sample_input, model_output=sample_output)

# 4. Log and Register to Unity Catalog
mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run() as run:
    # Log the model
    model_info = mlflow.prophet.log_model(
        pr_model=model,
        name="model",
        registered_model_name="workspace.default.sarasota_live_forecaster",
        signature=signature,
        input_example=sample_input 
    )
    
print(f"Gold Model successfully logged WITH signature! Version URI: {model_info.model_uri}")

21:19:40 - cmdstanpy - INFO - Chain [1] start processing
21:19:41 - cmdstanpy - INFO - Chain [1] done processing
/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
🔗 View Logged Model at: https://dbc-724c7636-740c.cloud.databricks.com/ml/experiments/2945210967405077/models/m-547ddcdf40e1422

Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

Gold Model successfully logged WITH signature! Version URI: models:/m-547ddcdf40e14222a859ace8a2c9c801


🔗 Created version '1' of model 'workspace.default.sarasota_live_forecaster': https://dbc-724c7636-740c.cloud.databricks.com/explore/data/models/workspace/default/sarasota_live_forecaster/version/1?o=7474644886349277
